In [1]:
# Single-cell Google Colab example
%pip install -q -U google-genai

import json
import time
from getpass import getpass
from google import genai

# Enter your key securely; it is not displayed.
#client = genai.Client(api_key=getpass("Enter your Gemini API key: "))

In [2]:
from google import genai
from google.colab import userdata

#API_KEY = put the new api key here created latest
api_key = userdata.get("GOOGLE_API_KEY")
client = genai.Client(api_key=api_key)

In [3]:
MODEL = "gemini-flash-latest"
TEMPERATURE_LIMIT = 80  # Illustrative threshold in degrees Celsius
CHECK_INTERVAL = 2     # Seconds between simulated readings

# Simulated readings: normal -> overheating -> recovery
sensor_readings = [62, 68, 75, 83, 88, 76]


class ProactiveMaintenanceAgent:
    def __init__(self):
        self.history = []
        self.alert_active = False

    def analyse_with_gemini(self):
        """Ask the LLM to interpret the observed incident."""
        prompt = f"""
You are a factory maintenance assistant.

Machine: Conveyor motor M-101
Temperature readings in chronological order:
{json.dumps(self.history)}

The demonstration alert threshold is {TEMPERATURE_LIMIT} degrees Celsius.
These are simulated readings, and no other sensor data is available.

Write a short maintenance alert containing:
1. What the readings show.
2. Possible causes, clearly labelled as hypotheses.
3. Three checks a qualified maintenance technician could perform.

Do not claim a confirmed diagnosis or that any action was performed.
"""
        response = client.models.generate_content(
            model=MODEL,
            contents=prompt,
        )
        return response.text or "No text recommendation returned."

    def observe(self, temperature):
        """Observe -> detect -> analyse -> alert -> remember."""
        self.history.append(temperature)
        print(f"\nSensor: M-101 temperature = {temperature} °C")

        if temperature >= TEMPERATURE_LIMIT:
            if self.alert_active:
                print("Monitoring: overheating incident already reported.")
                return

            # Issue the basic alert even if the LLM request fails.
            print("🚨 AUTOMATIC ALERT: Temperature threshold exceeded.")

            try:
                recommendation = self.analyse_with_gemini()
            except Exception as error:
                print(
                    f"Gemini request failed ({type(error).__name__}). "
                    "The temperature alert remains active."
                )
                # Leave this False so the next high reading retries.
                return

            print("\nGemini maintenance recommendation:")
            print(recommendation)
            self.alert_active = True

        elif self.alert_active:
            print("✅ Temperature is below the threshold. Alert cleared.")
            self.alert_active = False
        else:
            print("Monitoring: temperature below the alert threshold.")


agent = ProactiveMaintenanceAgent()

print("Agent started. Monitoring automatically...")

try:
    for index, temperature in enumerate(sensor_readings):
        agent.observe(temperature)

        if index < len(sensor_readings) - 1:
            time.sleep(CHECK_INTERVAL)
finally:
    client.close()

print("\nDemonstration completed.")

Agent started. Monitoring automatically...

Sensor: M-101 temperature = 62 °C
Monitoring: temperature below the alert threshold.

Sensor: M-101 temperature = 68 °C
Monitoring: temperature below the alert threshold.

Sensor: M-101 temperature = 75 °C
Monitoring: temperature below the alert threshold.

Sensor: M-101 temperature = 83 °C
🚨 AUTOMATIC ALERT: Temperature threshold exceeded.

Gemini maintenance recommendation:
**Maintenance Alert: Conveyor Motor M-101**

**1. Observations from Readings**
Chronological temperature readings: 62°C, 68°C, 75°C, 83°C.  
The data shows a continuous upward temperature trend, with the latest reading (83°C) breaching the 80°C alert threshold. 

**2. Potential Causes (Hypotheses)**
*Because no other sensor data is available, these are unconfirmed hypotheses:*
* **Hypothesis A:** Mechanical overload or binding along the conveyor line causing the motor to work beyond rated capacity.
* **Hypothesis B:** Bearing degradation or inadequate lubrication increas

In [8]:
# Re-initialize the client since it was closed in the previous cell's finally block
client = genai.Client(api_key=api_key)

test_prompt = "Hello! What model version are you?"

response = client.models.generate_content(
    model=MODEL,
    contents=test_prompt,
)

print("Requested model:", MODEL)
print("Actual model used:", response.model_version)
print("\nResponse text:\n", response.text)

Requested model: gemini-flash-latest
Actual model used: gemini-3.8-flash

Response text:
 Hello! I am Gemini 3.8 Flash, a large language model built by Google. How can I help you today?
